# Solving the CVRP using a Multimodal Genetic Algorithm (GA)

In this notebook, we will explore and evaluate different multimodal Genetic Algorithm (GA) configurations to solve the Capacitated Vehicle Routing Problem (CVRP).

## Introduction & Environment Setup

The Capacitated Vehicle Routing Problem (CVRP) is a classic logistics challenge where a fleet of vehicles with limited capacity must service a set of geographically dispersed customers. In real-world scenarios, finding a single 'optimal' route is often insufficient due to unforeseen disruptions, such as traffic or road closures. Therefore, our primary goal is to discover multiple, highly diverse routing alternatives of high quality for each problem instance (a multimodal approach).

In this initial section, we set up our experimental framework. We will load our dataset of CVRP instances and initialize our evolutionary engine and orchestrator classes designed to automate the execution, evaluation, and data logging of the genetic algorithms we will test.

## Why use a Genetic Algorithm?

Genetic Algorithms are powerful, bio-inspired metaheuristics based on the principles of Darwinian evolution and natural selection. Rather than constructing a single solution step-by-step probabilistically, a GA maintains a **population of diverse candidate routes**. 

Through successive generations, these candidate solutions evolve by exchanging genetic material (**crossover**) to inherit good topological traits from parent routes, and by introducing random variations (**mutation**) to explore new areas of the map. This population-based approach makes GAs exceptionally robust global searchers. It allows the algorithm to simultaneously explore multiple regions of the solution space, making it a naturally strong candidate for multimodal optimization where maintaining diverse routing strategies is the ultimate goal.

## Which algorithms are we going to evaluate?

To ensure a rigorous comparison with the swarm intelligence models, we have implemented a highly modular GA engine. We will evaluate our evolutionary approach across two key strategic dimensions to observe their impact on routing efficiency and diversity:

1. **Refinement & exploitation Strategies:**
    * **Pure Genetic Algorithm (Baseline):** Relies solely on classic evolutionary operators (OX1 Crossover and Swap Mutation) to explore the solution space. It acts as our control to measure raw evolutionary search power without local assistance.
    * **Memetic Algorithm (GA + Local Search):** A hybrid approach where a fast local search operator (such as 2-opt) is applied to the offspring. This acts as a localized learning step, allowing candidate routes to intelligently 'untangle' crossed paths before their fitness is evaluated, vastly accelerating convergence.

2. **Multimodal Diversity Techniques:**
    * **Standard Generational (Unimodal):** The algorithm prioritizes absolute fitness during replacement, naturally converging the entire population towards a single 'Global Best' route (risking premature convergence).
    * **Deterministic Crowding (Multimodal):** The algorithm actively protects diversity during the replacement phase using a Jaccard Distance threshold. Offspring only compete against structurally similar individuals. This forces the population to maintain multiple isolated "niches", yielding our required 3 distinct routing alternatives.

In [ ]:
# Libraries to use
import os # OS library
import time

# Core CVRP components shared across all metaheuristics
from src.common.parser import Parser
from src.common.problem import CVRPProblem

# Our custom evolutionary engine
from src.ga.multimodal_genetic_algorithm import MultimodalGeneticAlgorithm

## Chromosome Decoding & Local Search Optimization

Unlike ACO, which probabilistically constructs routes step-by-step, our Genetic Algorithm operates on a "giant tour" permutation—a single, continuous sequence of all clients without depot markers. To evaluate these permutations against the strict physical constraints of the CVRP (vehicle capacity), we employ a deterministic decoding strategy. Furthermore, to ensure geometrical efficiency and competitiveness, we elevate our baseline GA into a **Memetic Algorithm** by hybridizing the evolutionary process with an aggressive local search operator.

### Giant Tour Decoding & Implicit Feasibility

Dealing with capacity limits is a major challenge in evolutionary computation. If crossover operators had to constantly monitor vehicle capacities, the generation of valid offspring would be mathematically restrictive and highly inefficient. Instead, we implemented an **Implicit Feasibility (Split) Strategy**:

* **Dynamic Depot Insertion (Hard Constraints):** Our genetic operators (OX1 Crossover and Swap Mutation) only manage the relative ordering of clients, ignoring vehicle capacity entirely. During the decoding phase, the algorithm sequentially reads the permutation, accumulating customer demand. The moment adding the next customer would exceed the maximum vehicle capacity, the algorithm dynamically inserts a return trip to the central depot, resets the load, and dispatches a new vehicle. 
* This elegantly guarantees that **100% of the evaluated solutions are valid** and strictly adhere to capacity limits, completely bypassing the need for complex, heavy penalty functions (Soft Constraints) that can distort the evolutionary fitness landscape.

### Memetic Refinement: Windowed 2-Opt Local Search

While pure evolutionary operators are excellent at identifying promising global regions (exploration), they often struggle with local geometric inefficiencies, frequently leaving "knots" or crossed paths within individual vehicle routes. 

To bridge this gap, our engine applies a **Windowed 2-opt Local Search** to a percentage of the newly generated offspring. This memetic operator scans localized segments of the chromosome, systematically reversing subsets of nodes to untangle crossed edges. By restricting the 2-opt search to a sliding window rather than the entire route, we avoid extreme computational overhead ($O(N^2)$), achieving a highly efficient balance between global evolutionary exploration and localized exploitation.

In [ ]:
# Initialize the environment pointing to the common datasets folder
data_path = './data/X-n106-k14.vrp'  # Change to loop through 'data' folder if executing multiple

print(f"Loading data file: {data_path} ...")
vrp_parser = Parser(data_path)
nodes, demands, capacity = vrp_parser.parse()

# Instantiate the shared problem state
cvrp_instance = CVRPProblem(nodes, demands, capacity)

print(f"Environment ready. Successfully loaded CVRP instance: {len(nodes)} nodes, Vehicle Capacity: {capacity}")

In [ ]:
# Select a small instance for visual demonstration
import os
import matplotlib.pyplot as plt

from src.common.parser import Parser
from src.common.problem import CVRPProblem

sample_instance: str = 'X-n106-k14.vrp'
# Assuming the notebook is at the root or 'experiments' folder
file_path: str = os.path.join('./data', sample_instance) 

# Parse the topology of the terrain
sample_parser: Parser = Parser(file_path)
nodes, demands, capacity = sample_parser.parse()
sample_problem: CVRPProblem = CVRPProblem(nodes, demands, capacity)

# Separate the coordinates for plotting
depot_id: int = sample_problem.depot_id
depot_coord = nodes[depot_id]

# Extract customers' coordinates
customers_x = [coords[0] for node_id, coords in nodes.items() if node_id != depot_id]
customers_y = [coords[1] for node_id, coords in nodes.items() if node_id != depot_id]

# Draw the map
plt.figure(figsize=(8, 6))
# Try to use seaborn style if available, otherwise default
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    pass

# Plot the customers and depot
plt.scatter(customers_x, customers_y, c='blue', s=30, alpha=0.6, edgecolors='k', label='Customers')
plt.scatter(depot_coord[0], depot_coord[1], c='red', marker='s', s=120, edgecolors='k', label='Depot Central', zorder=5)

plt.title(f"CVRP Topology: {sample_instance}", fontweight='bold')
plt.xlabel("Coordinate X")
plt.ylabel("Coordinate Y")
plt.legend()
plt.show()

In [ ]:
# --- EXPERIMENT: Pure GA vs Memetic GA ---
import time

# Hiperparámetros para la demostración visual
DEMO_POP_SIZE = 50
DEMO_GENERATIONS = 150
DEMO_MUTATION = 0.1

print('--- Testing Raw Evolutionary Search (No 2-opt) ---')
ga_pure = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=DEMO_POP_SIZE)

start_time = time.time()
# use_local_search=False por defecto
top3_pure, hist_pure = ga_pure.run(generations=DEMO_GENERATIONS, mutation_rate=DEMO_MUTATION)
pure_time = time.time() - start_time
best_pure_route, best_pure_cost = top3_pure[0]


print('\n--- Testing Enhanced Memetic Search (Windowed 2-opt) ---')
# CAMBIO AQUÍ: Usamos cvrp_instance
ga_memetic = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=DEMO_POP_SIZE)

start_time = time.time()
# Activamos el interruptor memético
top3_memetic, hist_memetic = ga_memetic.run(
    generations=DEMO_GENERATIONS, 
    mutation_rate=DEMO_MUTATION, 
    use_local_search=True
)
memetic_time = time.time() - start_time
best_memetic_route, best_memetic_cost = top3_memetic[0]

print("\n--- PERFORMANCE SUMMARY ---")
print(f"Pure GA Cost: {best_pure_cost:.2f} (Time: {pure_time:.2f}s)")
print(f"Memetic GA Cost: {best_memetic_cost:.2f} (Time: {memetic_time:.2f}s)")
print(f"Improvement: {best_pure_cost - best_memetic_cost:.2f} distance units!")

In [ ]:
# --- VISUALIZATION: The impact of Local Search ---
import numpy as np

def plot_cvrp_route(ax, route, problem, title):
    """Auxiliary function to draw a CVRP route with pastel colors per vehicle."""
    depot_x, depot_y = problem.nodes[problem.depot_id]
    
    # Split the giant tour into individual vehicle routes
    vehicle_routes = []
    current_route = []
    for node in route:
        if node == problem.depot_id:
            if current_route:
                vehicle_routes.append(current_route)
                current_route = []
        else:
            current_route.append(node)
    if current_route:
        vehicle_routes.append(current_route)

    # Use a distinct color map for the vehicles
    colors = plt.cm.tab20(np.linspace(0, 1, len(vehicle_routes)))
    
    for i, sub_route in enumerate(vehicle_routes):
        # Add depot at start and end to close the loop
        full_path = [problem.depot_id] + sub_route + [problem.depot_id]
        xs = [problem.nodes[n][0] for n in full_path]
        ys = [problem.nodes[n][1] for n in full_path]
        
        ax.plot(xs, ys, marker='o', markersize=4, color=colors[i], alpha=0.7, label=f'Vehicle {i+1}')
        
    # Draw the central depot
    ax.plot(depot_x, depot_y, marker='s', color='red', markersize=10, label='Central Depot', zorder=5)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Coordinate X')
    ax.set_ylabel('Coordinate Y')
    ax.set_facecolor('#eaeaf2')
    ax.grid(color='white', linestyle='-', linewidth=1)

# Create a dual-plot figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

# Top Plot: Pure GA
# CAMBIO AQUÍ: Usamos cvrp_instance en lugar de sample_problem
plot_cvrp_route(ax1, best_pure_route, cvrp_instance, 
                f'Raw Evolutionary Decoding (No 2-opt applied: {best_pure_cost:.2f})')

# Bottom Plot: Memetic GA
# CAMBIO AQUÍ: Usamos cvrp_instance en lugar de sample_problem
plot_cvrp_route(ax2, best_memetic_route, cvrp_instance, 
                f'Enhanced Memetic Decoding (Windowed 2-opt applied: {best_memetic_cost:.2f})')

# Add a single legend outside the plots
handles, labels = ax2.get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', bbox_to_anchor=(1.15, 0.5), frameon=False)

plt.tight_layout()
plt.show()

## Multimodal Techniques: Overcoming Genetic Drift

The probabilistic nature of a Genetic Algorithm naturally generates a high degree of structural diversity during the initial generations. As the completely random initial population begins to cross over and mutate, it explores a vast amount of the solution space.

However, this initial diversity is fleeting. A standard GA is driven by intense selection pressure: the fittest individuals (shortest routes) are exponentially more likely to be selected as parents. Without explicit niching rules, this leads to a phenomenon known as **Genetic Drift** or **Premature Convergence**. The genetic material of the global best solution quickly dominates the entire gene pool, effectively wiping out alternative routing strategies and forcing the entire population to converge into a single, unimodal consensus path.

To systematically guarantee the discovery and preservation of multiple routing alternatives, we must introduce explicit Multimodal approaches. Instead of relying on standard generational replacement, we actively alter the algorithm's survival mechanics to force the population to deliberately divide its search efforts, conquer new geographical zones, and stably maintain multiple high-quality niches simultaneously.

### Deterministic Crowding (Active Niching)

To achieve multimodality, we implemented a highly aggressive replacement strategy known as **Deterministic Crowding**, driven by **Jaccard Distance** edge comparisons. 

Unlike passive archiving (which simply filters solutions at the very end), our GA enforces diversity *during* the evolutionary process. When a new offspring is generated, it does not simply replace the weakest global individual. Instead, its topology (edges) is compared against the existing population. If the offspring is structurally too similar to an existing solution (falling below our `similarity_threshold`), they are forced to compete for the exact same "niche." 

This acts as a hard mathematical "wall" against genetic homogenization, ensuring that the final population is naturally clustered into distinct, high-quality topographical niches. Let's execute the Memetic GA and extract our Top 3 diverse routing alternatives:

In [ ]:
# --- EXPERIMENT: Multimodal Niche Extraction ---
import time
from typing import List, Tuple

# Hyperparameters targeting deep multimodal exploration
MULTIMODAL_POP_SIZE: int = 100
MULTIMODAL_GENERATIONS: int = 200
MULTIMODAL_MUTATION: float = 0.1
SIMILARITY_THRESHOLD: float = 0.20  # Require at least 20% topological difference between niches

print(f"--- Launching Multimodal Memetic GA (Threshold: {SIMILARITY_THRESHOLD}) ---")
# Instantiate the engine using our shared CVRP instance
ga_multimodal = MultimodalGeneticAlgorithm(cvrp_instance, pop_size=MULTIMODAL_POP_SIZE)

start_time_multi: float = time.time()

# Execute the algorithm (Memetic + Crowding)
top3_niches: List[Tuple[List[int], float]]
history_multi: List[float]

top3_niches, history_multi = ga_multimodal.run(
    generations=MULTIMODAL_GENERATIONS, 
    mutation_rate=MULTIMODAL_MUTATION, 
    similarity_threshold=SIMILARITY_THRESHOLD,
    use_local_search=True # Utilizing our Windowed 2-opt for geometric efficiency
)

execution_time_multi: float = time.time() - start_time_multi

print("\n--- MULTIMODAL EXTRACTION COMPLETE ---")
print(f"Execution Time: {execution_time_multi:.2f} seconds")
for i, (route, cost) in enumerate(top3_niches):
    validity: str = "Valid" if cvrp_instance.is_route_valid(route) else "INVALID (Capacity Overflow)"
    print(f"🏆 Niche {i+1} | Cost: {cost:.2f} | Status: {validity}")

In [ ]:
# --- VISUALIZATION: The 3 Distinct Topological Niches ---
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.axes import Axes

def plot_single_niche(ax: Axes, route: List[int], problem: CVRPProblem, title: str) -> None:
    """
    Plots a single CVRP route on the provided matplotlib Axes.
    Strictly type-hinted for production-grade code quality.
    """
    depot_x: float
    depot_y: float
    depot_x, depot_y = problem.nodes[problem.depot_id]
    
    # Segment the giant tour into independent vehicle trajectories
    vehicle_routes: List[List[int]] = []
    current_route: List[int] = []
    
    for node in route:
        if node == problem.depot_id:
            if current_route:
                vehicle_routes.append(current_route)
                current_route = []
        else:
            current_route.append(node)
            
    if current_route:
        vehicle_routes.append(current_route)

    # Dynamic color mapping for varying fleet sizes
    colors = plt.cm.tab20(np.linspace(0, 1, len(vehicle_routes)))
    
    for i, sub_route in enumerate(vehicle_routes):
        # Close the loop by prepending and appending the depot
        full_path: List[int] = [problem.depot_id] + sub_route + [problem.depot_id]
        
        xs: List[float] = [problem.nodes[n][0] for n in full_path]
        ys: List[float] = [problem.nodes[n][1] for n in full_path]
        
        ax.plot(xs, ys, marker='o', markersize=3, color=colors[i], alpha=0.7)
        
    # Overlay the central depot
    ax.plot(depot_x, depot_y, marker='s', color='red', markersize=8, zorder=5)
    
    # Axis styling
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_facecolor('#eaeaf2')
    ax.grid(color='white', linestyle='-', linewidth=0.5)
    ax.set_xticks([]) # Remove ticks for cleaner visual presentation
    ax.set_yticks([])

# Ensure we have at least 3 routes to plot, otherwise pad with empty
while len(top3_niches) < 3:
    top3_niches.append(([], 0.0))

# Initialize a 1x3 horizontal grid
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Top 3 Structurally Distinct Routing Alternatives (Jaccard Threshold: {SIMILARITY_THRESHOLD * 100}%)", 
             fontsize=14, fontweight='bold', y=1.05)

# Render each niche
for i, ax in enumerate(axes):
    if top3_niches[i][0]: # If route exists
        route, cost = top3_niches[i]
        plot_single_niche(ax, route, cvrp_instance, f"Niche {i+1} Topology\nTotal Cost: {cost:.2f}")
    else:
        ax.set_title(f"Niche {i+1} (Not Found)")
        ax.axis('off')

plt.tight_layout()
plt.show()

## Evolutionary Convergence & Statistical Analysis

To evaluate the stability and learning rate of our Memetic Algorithm, we first analyze its convergence history. A steep initial drop indicates rapid global exploration, while a plateau in later generations suggests that the algorithm has successfully fine-tuned the solutions within their respective niches (exploitation).

Furthermore, unlike algorithms that return a massive unfiltered archive of similar solutions, our Deterministic Crowding approach distills the population into exactly 3 structurally distinct, highly optimized niches. Below, we visualize the evolutionary learning curve alongside the fitness distribution of our final multimodal alternatives.

In [ ]:
# --- ANALYTICS: Convergence and Niche Distribution ---
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Tuple

def plot_ga_analytics(history: List[float], niches: List[Tuple[List[int], float]]) -> None:
    """Renders the convergence curve and statistical distribution of the found niches."""
    
    # 1. Convergence Curve
    plt.figure(figsize=(12, 4))
    plt.plot(history, color='forestgreen', linewidth=2, linestyle='-')
    plt.title("Memetic GA: Convergence History", fontweight='bold')
    plt.xlabel("Generation (Iteration)")
    plt.ylabel("Best Fitness (Distance)")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 2. Statistical Analysis of Final Niches
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Statistical Analysis of Extracted Multimodal Niches", fontweight='bold')
    
    niche_costs: List[float] = [cost for _, cost in niches if cost > 0]
    labels: List[str] = [f"Niche {i+1}" for i in range(len(niche_costs))]
    
    # Left: Bar chart of Costs
    bars = ax1.bar(labels, niche_costs, color=['#5cb85c', '#5bc0de', '#f0ad4e'], edgecolor='black')
    ax1.set_title("Fitness by Niche")
    ax1.set_ylabel("Route Cost")
    ax1.set_ylim(min(niche_costs) * 0.95, max(niche_costs) * 1.05) # Zoom in for detail
    
    # Right: Boxplot to show dispersion context
    # We use a boxplot to align with the ACO team's visual style
    ax2.boxplot(niche_costs, vert=False, patch_artist=True, 
                boxprops=dict(facecolor='#d9534f', color='black'),
                medianprops=dict(color='white', linewidth=2))
    ax2.set_title("Fitness Dispersion (Boxplot)")
    ax2.set_xlabel("Route Cost")
    ax2.set_yticks([]) # Hide y-axis labels for boxplot

    plt.tight_layout()
    plt.show()

# Call the analytics plotter using the data from the previous cell
plot_ga_analytics(history_multi, top3_niches)

In [ ]:
# --- ANALYTICS: Gene Pool Consensus Heatmap ---
import matplotlib.pyplot as plt
import numpy as np

def plot_gene_pool_heatmap(problem: 'CVRPProblem', niches: List[Tuple[List[int], float]]) -> None:
    """
    Generates a heatmap of edge frequencies (Gene Consensus) across the final niches.
    Acts as the GA counterpart to the ACO Pheromone Matrix.
    """
    num_nodes: int = len(problem.nodes)
    consensus_matrix: np.ndarray = np.zeros((num_nodes, num_nodes))
    
    # Tally the edges used in all extracted niches
    valid_niches = [route for route, cost in niches if route]
    
    for route in valid_niches:
        # Reconstruct full paths with depots
        vehicle_routes = []
        current = []
        for node in route:
            if node == problem.depot_id:
                if current: vehicle_routes.append(current)
                current = []
            else:
                current.append(node)
        if current: vehicle_routes.append(current)
            
        for sub_route in vehicle_routes:
            full_path = [problem.depot_id] + sub_route + [problem.depot_id]
            for i in range(len(full_path) - 1):
                u, v = full_path[i], full_path[i+1]
                # Increment both directions for an undirected interpretation
                consensus_matrix[u][v] += 1
                consensus_matrix[v][u] += 1
                
    # Plotting
    plt.figure(figsize=(8, 6))
    
    # We use the 'magma' colormap to simulate the glowing dark-background style of ACO
    im = plt.imshow(consensus_matrix, cmap='magma', interpolation='nearest', aspect='auto')
    
    plt.colorbar(im, label='Frequency of Edge Occurrence (Consensus Level)')
    plt.title("Gene Pool Consensus Matrix (Edge Frequencies)", fontweight='bold', color='white')
    
    # Styling for a "dark mode" heatmap
    ax = plt.gca()
    ax.set_facecolor('black')
    fig = plt.gcf()
    fig.patch.set_facecolor('#2b2b2b') # Dark grey background
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    
    plt.xlabel("From Node ID")
    plt.ylabel("To Node ID")
    
    plt.tight_layout()
    plt.show()

# Render the Heatmap
plot_gene_pool_heatmap(cvrp_instance, top3_niches)